# 03 · Fuga de datos: por qué tu modelo miente

**Módulo 2 · Sesión 5** — Ingeniería de características

## Objetivos

La **fuga de datos** (*data leakage*) ocurre cuando información que no estará disponible en
el momento de predecir se cuela en el entrenamiento. El resultado es siempre el mismo:
métricas excelentes en el experimento y un modelo que fracasa en producción.

Es el error más costoso del Machine Learning aplicado, y el más difícil de detectar,
porque **no produce ningún error**: produce buenas noticias.

En este notebook no vamos a describir la fuga: vamos a **medirla**.

1. El caso extremo: aprender de ruido puro.
2. Cuánto infla cada tipo de fuga, medido con error estándar. El resultado sorprende.
3. Fuga que viene dentro del propio dataset.
4. Fuga temporal y fuga por grupos.
5. La solución: `Pipeline`.

## Paquetes

`numpy`, `pandas`, `matplotlib`, `scikit-learn`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-2-datos-caracteristicas/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEMILLA = 42

## 1. El experimento imposible

Empecemos por el caso más extremo, porque deja la idea grabada.

Vamos a construir un dataset donde **es imposible aprender nada**:

- 200 observaciones y 10.000 características, todas **ruido aleatorio puro**.
- Una etiqueta binaria, también **completamente aleatoria**.

No hay ninguna relación entre `X` e `y`. Ninguna. Cualquier modelo honesto debería acertar
alrededor del **50 %**, como lanzar una moneda.

In [ ]:
rng = np.random.default_rng(SEMILLA)

n_observaciones = 200
n_caracteristicas = 10_000

X_ruido = rng.normal(size=(n_observaciones, n_caracteristicas))
y_ruido = rng.integers(0, 2, size=n_observaciones)

print(f"X: {X_ruido.shape}  ·  y: {y_ruido.shape}")
print(f"Proporción de clase 1: {y_ruido.mean():.2f}")
print("\nCorrelación real entre X e y: NINGUNA (ambos son aleatorios independientes)")

### La forma incorrecta

Un procedimiento que parece razonable y se ve constantemente:

1. "Tengo 10.000 características, son demasiadas. Selecciono las 20 más relacionadas con el
   objetivo."
2. "Ahora evalúo con validación cruzada, que es la forma rigurosa de evaluar."

El error está en el orden: la selección mira **todos** los datos, incluidos los que después
harán de validación.

In [ ]:
selector = SelectKBest(score_func=f_classif, k=20)
X_seleccionado = selector.fit_transform(X_ruido, y_ruido)   # <-- mira TODO el dataset

modelo = LogisticRegression(max_iter=1000)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
puntajes_mal = cross_val_score(modelo, X_seleccionado, y_ruido, cv=cv, scoring="accuracy")

print(f"Accuracy por pliegue: {puntajes_mal.round(3)}")
print(f"Accuracy media:       {puntajes_mal.mean():.3f}")

**Sobre ruido puro.** El modelo "aprendió" a clasificar datos que no contienen ninguna
información. Si esto fuera un proyecto real, aquí es donde alguien escribe en el informe que
el modelo funciona.

### Qué pasó exactamente

Con 10.000 características aleatorias, **por puro azar** algunas se parecerán a la etiqueta
en estas 200 observaciones concretas. `SelectKBest` las encontró — que es su trabajo — pero
al mirar el dataset completo, esas coincidencias incluyen las filas que luego se usan para
validar.

La validación cruzada ya no evalúa nada: las características fueron elegidas *sabiendo* las
respuestas del conjunto de validación.

In [ ]:
puntajes_f = selector.scores_
print(f"Estadístico F: máximo {np.nanmax(puntajes_f):.2f}, mediana {np.nanmedian(puntajes_f):.2f}")
print(f"Características con p-valor < 0.01 por azar: {(selector.pvalues_ < 0.01).sum()} de {n_caracteristicas}")
print(f"Esperadas por azar con alfa=0.01:            {int(0.01 * n_caracteristicas)}")

Ahí está la explicación: con 10.000 pruebas al 1 % de significancia, unas 100 salen
"significativas" solo por azar. Es el problema de las comparaciones múltiples, convertido en
fuga de datos.

### La forma correcta

La selección de características **es parte del modelo**, no un paso previo. Debe ocurrir
dentro de cada pliegue de la validación cruzada, usando solo los datos de entrenamiento de
ese pliegue. Eso es exactamente lo que hace un `Pipeline`.

In [ ]:
flujo_correcto = Pipeline(
    [
        ("seleccion", SelectKBest(score_func=f_classif, k=20)),
        ("modelo", LogisticRegression(max_iter=1000)),
    ]
)
puntajes_bien = cross_val_score(flujo_correcto, X_ruido, y_ruido, cv=cv, scoring="accuracy")

print(f"Accuracy por pliegue: {puntajes_bien.round(3)}")
print(f"Accuracy media:       {puntajes_bien.mean():.3f}")

In [ ]:
fig, eje = plt.subplots(figsize=(7, 4))
eje.bar(
    ["Selección fuera\ndel pipeline\n(FUGA)", "Selección dentro\ndel pipeline\n(correcto)"],
    [puntajes_mal.mean(), puntajes_bien.mean()],
    color=["tab:red", "tab:green"],
)
eje.axhline(0.5, color="black", linestyle="--", linewidth=1, label="Azar (0.50)")
eje.set_ylabel("Accuracy (validación cruzada)")
eje.set_ylim(0, 1)
eje.set_title("Mismo dato: ruido puro. Mismo modelo. Distinto orden de operaciones.")
eje.legend()
plt.tight_layout()
plt.show()

Con el orden correcto, el resultado es el que debe ser: **azar**. El modelo dice la verdad,
que es que no hay nada que aprender.

> **La regla.** Cualquier paso que *aprenda algo de los datos* —una media, una desviación,
> una lista de características, una mediana para imputar— debe aprenderlo **solo del
> conjunto de entrenamiento**.


## 2. ¿Todas las fugas son igual de graves?

El caso anterior es extremo a propósito. Ahora la pregunta práctica: de los errores que de
verdad se cometen a diario, **¿cuánto infla cada uno la métrica?**

Se suele enseñar que estandarizar antes de partir los datos es un pecado grave. Vamos a
medirlo en lugar de repetirlo.

Compararemos tres formas de hacer las cosas mal contra su versión correcta:

| Fuga | Versión incorrecta | Versión correcta |
|---|---|---|
| **Escalado** | `StandardScaler` sobre todo el dataset, luego partir | Escalador dentro del `Pipeline` |
| **Imputación** | `SimpleImputer` sobre todo el dataset, luego partir | Imputador dentro del `Pipeline` |
| **Selección** | `SelectKBest` sobre todo el dataset, luego partir | Selector dentro del `Pipeline` |

La clave metodológica: **una sola partición no sirve para medir esto**. La diferencia entre
dos particiones al azar es mayor que el efecto que buscamos. Hay que promediar sobre muchas
repeticiones y reportar el error estándar.

In [ ]:
def generar_datos(n, p, semilla, ruido=2.0):
    """Datos con señal real y escalas muy distintas entre variables."""
    rng = np.random.default_rng(semilla)
    X = rng.normal(size=(n, p)) * rng.uniform(1, 100, size=p)
    pesos = rng.normal(size=p)
    y = ((X / X.std(axis=0)) @ pesos + rng.normal(scale=ruido, size=n) > 0).astype(int)
    return X, y


X_demo, y_demo = generar_datos(n=300, p=8, semilla=SEMILLA)
print(f"X: {X_demo.shape}  ·  proporción de clase 1: {y_demo.mean():.2f}")
print(f"Desviación de cada variable: {X_demo.std(axis=0).round(1)}")
print("\nLas escalas son deliberadamente dispares: el escenario donde el escalado más importa.")

### Un intento con una sola partición

Así es como casi todo el mundo comprobaría el efecto del escalado:

In [ ]:
X_mal = StandardScaler().fit_transform(X_demo)          # <-- ve todo, prueba incluida
a, b, ya, yb = train_test_split(X_mal, y_demo, test_size=0.3, random_state=SEMILLA, stratify=y_demo)
acc_mal = KNeighborsClassifier(5).fit(a, ya).score(b, yb)

a2, b2, ya2, yb2 = train_test_split(X_demo, y_demo, test_size=0.3, random_state=SEMILLA, stratify=y_demo)
flujo = Pipeline([("escalar", StandardScaler()), ("modelo", KNeighborsClassifier(5))])
acc_bien = flujo.fit(a2, ya2).score(b2, yb2)

print(f"Escalando ANTES de partir:     {acc_mal:.4f}")
print(f"Escalando dentro del Pipeline: {acc_bien:.4f}")
print(f"Diferencia: {acc_mal - acc_bien:+.4f}")

Sale una diferencia pequeña. Pero **este número no significa nada**: con otra semilla podría
salir del signo contrario. Lo comprobamos midiendo en serio.

In [ ]:
def medir_sesgo(n, p, k_seleccion, frac_faltantes=0.30, repeticiones=100):
    """Sobreoptimismo medio de cada tipo de fuga, promediado sobre muchas particiones."""
    resultados = {"escalado": [], "imputacion": [], "seleccion": []}

    for r in range(repeticiones):
        X, y = generar_datos(n, p, semilla=1000 + r)
        rng_na = np.random.default_rng(2000 + r)
        parametros = dict(test_size=0.3, random_state=r, stratify=y)
        regresion = lambda: LogisticRegression(max_iter=1000)

        # --- Escalado ---
        Xm = StandardScaler().fit_transform(X)
        a, b, ya, yb = train_test_split(Xm, y, **parametros)
        mal = regresion().fit(a, ya).score(b, yb)
        a2, b2, ya2, yb2 = train_test_split(X, y, **parametros)
        bien = Pipeline([("s", StandardScaler()), ("m", regresion())]).fit(a2, ya2).score(b2, yb2)
        resultados["escalado"].append(mal - bien)

        # --- Imputación ---
        Xna = X.copy()
        Xna[rng_na.random(X.shape) < frac_faltantes] = np.nan
        Xi = SimpleImputer(strategy="mean").fit_transform(Xna)
        a, b, ya, yb = train_test_split(Xi, y, **parametros)
        mal = Pipeline([("s", StandardScaler()), ("m", regresion())]).fit(a, ya).score(b, yb)
        a2, b2, ya2, yb2 = train_test_split(Xna, y, **parametros)
        bien = Pipeline(
            [("i", SimpleImputer(strategy="mean")), ("s", StandardScaler()), ("m", regresion())]
        ).fit(a2, ya2).score(b2, yb2)
        resultados["imputacion"].append(mal - bien)

        # --- Selección de características ---
        Xs = SelectKBest(f_classif, k=k_seleccion).fit_transform(X, y)
        a, b, ya, yb = train_test_split(Xs, y, **parametros)
        mal = Pipeline([("s", StandardScaler()), ("m", regresion())]).fit(a, ya).score(b, yb)
        a2, b2, ya2, yb2 = train_test_split(X, y, **parametros)
        bien = Pipeline(
            [("sel", SelectKBest(f_classif, k=k_seleccion)), ("s", StandardScaler()), ("m", regresion())]
        ).fit(a2, ya2).score(b2, yb2)
        resultados["seleccion"].append(mal - bien)

    return {n_: (np.mean(v), np.std(v) / np.sqrt(len(v))) for n_, v in resultados.items()}


escenarios = [(60, 20, 5), (100, 50, 10), (200, 200, 10)]

print(f"{'n':>5} {'p':>5} {'k':>4} |{'escalado':>19}{'imputación':>19}{'selección':>19}")
print("-" * 76)
medidas = {}
for n, p, k in escenarios:
    m = medir_sesgo(n, p, k)
    medidas[(n, p)] = m
    fila = "".join(
        f"{m[c][0]:>+12.4f} ±{m[c][1]:.4f}" for c in ["escalado", "imputacion", "seleccion"]
    )
    print(f"{n:>5} {p:>5} {k:>4} |{fila}")

### El resultado, que probablemente no esperabas

- **Escalado: sesgo indistinguible de cero.** En los tres escenarios el efecto es del orden
  de su propio error estándar. No hay sobreoptimismo medible.
- **Imputación con la media: lo mismo.** Tampoco se detecta efecto.
- **Selección de características: entre 8 y 13 puntos de accuracy inflados**, y crece cuanto
  más se acerca $p$ a $n$.

La explicación está en **cuánta información sobre el objetivo se filtra**. Al escalar, del
conjunto de prueba se filtran dos números por variable (una media y una desviación), que
además no dicen nada sobre `y`. Al seleccionar características se filtra *qué variables se
parecen a la respuesta*: información directamente sobre el objetivo, y en cantidad
proporcional al número de variables candidatas.

In [ ]:
fig, eje = plt.subplots(figsize=(8, 4.5))
etiquetas = [f"n={n}, p={p}" for n, p in medidas]
posiciones = np.arange(len(etiquetas))
ancho = 0.25

for i, (clave, color) in enumerate(
    [("escalado", "tab:blue"), ("imputacion", "tab:orange"), ("seleccion", "tab:red")]
):
    valores = [medidas[k][clave][0] for k in medidas]
    errores = [medidas[k][clave][1] for k in medidas]
    eje.bar(posiciones + (i - 1) * ancho, valores, ancho, yerr=errores, capsize=3,
            label=clave, color=color)

eje.axhline(0, color="black", linewidth=0.8)
eje.set_xticks(posiciones)
eje.set_xticklabels(etiquetas)
eje.set_ylabel("Sobreoptimismo (accuracy inflada)")
eje.set_title("No todas las fugas son igual de graves")
eje.legend()
plt.tight_layout()
plt.show()

### Qué hacer con esto

**No** es una licencia para escalar antes de partir. Sigue siendo incorrecto, y meterlo en un
`Pipeline` no cuesta nada — así que hazlo siempre.

Lo que este experimento aporta es **sentido de la proporción**, que es justo lo que falta en
la mayoría de las advertencias sobre fuga de datos:

> Si un modelo funcionó en el experimento y fracasó en producción, el culpable casi nunca es
> el escalador. Busca primero: variables que no existirán en el momento de predecir,
> variables derivadas del objetivo, y selección de características hecha fuera de la
> validación.

Y una advertencia sobre el propio experimento: estos números valen para **este** escenario.
Con outliers extremos en el conjunto de prueba, con muestras de 20 observaciones, o con
transformaciones más agresivas que un `StandardScaler` (`PowerTransformer`, PCA,
discretización por cuantiles), el sesgo del preprocesamiento sí crece. La conclusión robusta
no es "el escalado da igual", sino **"la gravedad depende de cuánta información sobre el
objetivo se filtra"**.


## 3. La fuga que viene en el dataset

Volvamos al Titanic del notebook 01. Allí descubrimos que la columna `alive` es la variable
objetivo escrita en texto. Veamos qué pasa si nadie se da cuenta.

In [ ]:
titanic = pd.read_csv("../datos/titanic.csv")

X_fuga = pd.get_dummies(
    titanic[["pclass", "sex", "age", "fare", "alive"]].fillna({"age": titanic["age"].median()}),
    columns=["sex", "alive"],
    drop_first=True,
)
y_titanic = titanic["survived"]

acc_fuga = cross_val_score(
    LogisticRegression(max_iter=1000), X_fuga, y_titanic, cv=5, scoring="accuracy"
).mean()

X_limpio = pd.get_dummies(
    titanic[["pclass", "sex", "age", "fare"]].fillna({"age": titanic["age"].median()}),
    columns=["sex"],
    drop_first=True,
)
acc_limpio = cross_val_score(
    LogisticRegression(max_iter=1000), X_limpio, y_titanic, cv=5, scoring="accuracy"
).mean()

print(f"Accuracy CON la columna 'alive':  {acc_fuga:.4f}")
print(f"Accuracy SIN la columna 'alive':  {acc_limpio:.4f}")

**Accuracy perfecta.** Y un modelo que no sirve para nada: para predecir si un pasajero
sobrevivirá necesita saber si sobrevivió.

> **Señal de alarma.** Cuando una métrica sale sospechosamente buena, la explicación más
> probable no es que tu modelo sea genial: es que hay una fuga. Un resultado demasiado bueno
> merece más escrutinio que uno malo, no menos.

### Cómo detectarla

Un diagnóstico barato: revisar qué variables tienen una asociación desmedida con el objetivo.

In [ ]:
asociacion = X_fuga.apply(lambda columna: columna.corr(y_titanic)).abs().sort_values(ascending=False)
print("Correlación absoluta con el objetivo:")
print(asociacion.round(3).to_string())

Una correlación de 1.000 es un cartel luminoso. En un caso real no será exactamente 1, pero
una variable con una asociación mucho mayor que el resto siempre merece la pregunta: **¿esta
información existirá realmente en el momento de predecir?**

## 4. Otras dos fugas que hay que conocer

### Fuga temporal

Si los datos tienen orden cronológico, una partición aleatoria entrena con el futuro y
valida con el pasado.

| Situación | Problema |
|---|---|
| Predecir ventas de diciembre entrenando con datos de enero a marzo del año siguiente | El modelo ve el futuro |
| Predecir deserción usando el promedio del semestre completo | Ese dato no existe cuando hay que intervenir |
| Predecir readmisión hospitalaria usando consultas posteriores al alta | Se conocen después del alta |

**Solución:** partición temporal (entrenar con el pasado, validar con el futuro) y
`TimeSeriesSplit` en lugar de `KFold`. Lo veremos en la sesión 8.

### Fuga por grupos

Si un mismo sujeto aparece en varias filas y unas caen en entrenamiento y otras en prueba,
el modelo puede **reconocer al sujeto** en lugar de aprender el patrón.

Ejemplos: varias mediciones del mismo paciente, varias fotos de la misma persona, varios
semestres del mismo estudiante.

**Solución:** `GroupKFold` o `GroupShuffleSplit`, que garantizan que todas las filas de un
grupo caigan del mismo lado.

## 5. La solución general: `Pipeline`

Un `Pipeline` encadena transformaciones y modelo en un solo objeto que respeta la separación
por construcción:

- Al llamar a `.fit()`, cada paso hace `fit_transform` **solo con los datos de
  entrenamiento**.
- Al llamar a `.predict()`, cada paso hace únicamente `transform`, con lo aprendido antes.
- Dentro de `cross_val_score`, todo eso ocurre **por separado en cada pliegue**.

Es decir: usar `Pipeline` no es una cuestión de elegancia, es lo que hace que la evaluación
sea honesta. El notebook 04 lo construye en detalle.

In [ ]:
# Datos con valores faltantes, escalas dispares y variables irrelevantes: los tres problemas
# que resuelven los tres pasos del pipeline.
X_sucio, y_sucio = generar_datos(n=300, p=20, semilla=SEMILLA)
X_sucio = X_sucio.copy()
X_sucio[np.random.default_rng(SEMILLA).random(X_sucio.shape) < 0.20] = np.nan

flujo_completo = Pipeline(
    [
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
        ("seleccionar", SelectKBest(f_classif, k=5)),
        ("modelo", LogisticRegression(max_iter=1000)),
    ]
)
puntajes = cross_val_score(flujo_completo, X_sucio, y_sucio, cv=cv, scoring="accuracy")

print(f"Datos: {X_sucio.shape}, {np.isnan(X_sucio).mean():.0%} de valores faltantes")
print(f"Accuracy con validación cruzada honesta: {puntajes.mean():.4f} (±{puntajes.std():.4f})")

Los cuatro pasos —imputar, escalar, seleccionar y modelar— se ajustan **por separado en cada
uno de los cinco pliegues**, siempre con los datos de entrenamiento de ese pliegue. Ese es
todo el truco, y por eso el resultado es creíble.

## Lista de verificación contra fugas

Antes de creerte cualquier resultado:

- [ ] ¿Se partieron los datos **antes** de cualquier transformación que aprenda algo?
- [ ] ¿Está todo el preprocesamiento dentro de un `Pipeline`?
- [ ] ¿La selección de características ocurre **dentro** de la validación cruzada?
- [ ] ¿Cada variable predictora existirá en el momento real de predecir?
- [ ] ¿Alguna variable se calcula a partir del objetivo, aunque sea indirectamente?
- [ ] Si hay tiempo, ¿la partición respeta el orden cronológico?
- [ ] Si hay sujetos repetidos, ¿se agruparon con `GroupKFold`?
- [ ] ¿Hay alguna variable con una asociación desmedida con el objetivo?
- [ ] Si el resultado parece demasiado bueno, **¿lo investigaste?**

## Resumen

| Tipo de fuga | Cómo se cuela | Gravedad | Solución |
|---|---|---|---|
| Variable derivada del objetivo | Columna que codifica la respuesta | **Muy alta** (hasta accuracy = 1) | Revisar el diccionario de datos |
| Variable del futuro | Usar datos posteriores al hecho a predecir | **Muy alta** | Preguntar qué se sabe al predecir |
| Selección de características | Elegir variables mirando todo el dataset | **Muy alta** (+0.09 a +0.13 medido) | Selección dentro del `Pipeline` |
| Temporal | Partición aleatoria con datos ordenados | Alta | `TimeSeriesSplit` |
| Por grupos | Un sujeto en entrenamiento y prueba | Alta | `GroupKFold` |
| Imputación | Calcular medias sobre todo el dataset | Baja: no medible en el experimento | `SimpleImputer` en el `Pipeline` |
| Escalado | Calcular medias sobre todo el dataset | Baja: no medible en el experimento | `StandardScaler` en el `Pipeline` |

Las dos últimas siguen siendo incorrectas y se corrigen gratis con un `Pipeline`. Pero
cuando busques por qué un modelo falló en producción, empieza por arriba de la tabla.

## Para practicar

1. En el experimento de la sección 1, prueba con `k=5`, `k=50` y `k=200` características
   seleccionadas. ¿Cuándo es mayor el sobreoptimismo? Explica por qué.
2. Repite el experimento con 100 características en vez de 10.000. ¿Desaparece la fuga o
   solo se atenúa?
3. Construye un caso de fuga por grupos: genera datos donde cada "paciente" tenga 5
   mediciones, compara `KFold` con `GroupKFold` y mide la diferencia.
4. Sobre el Titanic, comprueba si `class` o `embark_town` producen fuga. ¿Por qué no, si
   también son columnas duplicadas?